# Notebook — Generative models, the concepts (in NumPy)

The big generative ideas — without a GPU. Each maps to a model in the theory:

- **Part A — a linear autoencoder *is* PCA.** Compress images to a few latent
  numbers and reconstruct; reconstruction sharpens as you keep more components.
- **Part B — generative sampling & interpolation.** Fit a Gaussian to the latent
  codes, **sample** new latents and decode → brand-new images; **interpolate**
  between two codes → a smooth morph. This is the VAE generative recipe in linear
  form.
- **Part C — the adversarial idea.** A 2D toy where a "generator" distribution
  chases the real one while a "discriminator" tries to separate them.

NumPy + Matplotlib only. For real VAEs / GANs / diffusion, use the MONAI
Generative Models tutorials (cited in the theory) in the `mic_soc` environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)

# --- a small image dataset: soft Gaussian blobs with varying position & width ---
N = 28
def make_blob(cx, cy, s):
    yy, xx = np.mgrid[0:N, 0:N]
    return np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2*s**2))

M = 300
cxs = rng.uniform(8, 20, M); cys = rng.uniform(8, 20, M); ss = rng.uniform(2.5, 5.0, M)
imgs = np.array([make_blob(cxs[i], cys[i], ss[i]) for i in range(M)])  # (M, N, N)
Xd = imgs.reshape(M, -1)                                                # (M, 784)
print('dataset:', imgs.shape, '-> vectors', Xd.shape)

fig, ax = plt.subplots(1, 6, figsize=(13, 2.4))
for j in range(6):
    ax[j].imshow(imgs[j], cmap='magma'); ax[j].axis('off')
fig.suptitle('sample images (blobs that vary in position and width)'); plt.show()

## Part A — a linear autoencoder is PCA

With a linear encoder/decoder and squared-error loss, the best autoencoder
projects onto the top principal components. So we compute PCA, **reconstruct**
each image from its top-`k` latent numbers, and watch quality rise with `k`. The
"latent code" is just the PCA scores; the decoder is the inverse projection.

In [ ]:
mu = Xd.mean(0)
Xc = Xd - mu
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)   # PCA via SVD; rows of Vt = components

def reconstruct(k):
    Z = Xc @ Vt[:k].T          # encode: top-k latent scores  (M, k)
    return (Z @ Vt[:k]) + mu   # decode: inverse projection   (M, 784)

idxs = rng.choice(M, 5, replace=False)
rows = [('original', Xd)] + [(f'k = {k}', reconstruct(k)) for k in (1, 3, 10)]
fig, ax = plt.subplots(len(rows), 5, figsize=(11, 9))
for r, (label, data) in enumerate(rows):
    for c, i in enumerate(idxs):
        ax[r, c].imshow(data[i].reshape(N, N), cmap='magma'); ax[r, c].axis('off')
    ax[r, 0].set_ylabel(label, rotation=0, ha='right', va='center', fontsize=11)
plt.tight_layout(); plt.show()

# variance spectrum: a few components explain almost everything
var = S**2 / np.sum(S**2)
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, 16), np.cumsum(var)[:15], 'o-')
plt.xlabel('# latent components'); plt.ylabel('cumulative variance explained')
plt.ylim(0, 1.02); plt.title('a handful of latents capture this dataset'); plt.show()

## Part B — generative sampling and interpolation

The generative move (the heart of a VAE): give the latent space a **distribution**.
We fit a Gaussian to the latent codes, then (1) **sample** new latents and decode
them into never-seen images, and (2) **interpolate** between two real images'
codes for a smooth morph — evidence the latent space is filled in, not full of holes.

In [ ]:
k = 6
Vk = Vt[:k]                 # decoder directions (k, 784)
scores = Xc @ Vk.T          # latent codes (M, k)
sd = scores.std(0)          # per-latent spread (fit a diagonal Gaussian)

# (1) SAMPLE new latents ~ N(0, diag(sd^2)) and decode -> new images
zsamp = rng.normal(0, 1, (8, k)) * sd
gen = (zsamp @ Vk) + mu
fig, ax = plt.subplots(1, 8, figsize=(14, 2.2))
for j in range(8):
    ax[j].imshow(np.clip(gen[j].reshape(N, N), 0, 1), cmap='magma'); ax[j].axis('off')
fig.suptitle('NEW images sampled from the Gaussian latent space (generation)'); plt.show()

# (2) INTERPOLATE between two real images' latent codes
a, b = rng.choice(M, 2, replace=False)
fig, ax = plt.subplots(1, 8, figsize=(14, 2.2))
for j, t in enumerate(np.linspace(0, 1, 8)):
    z = (1 - t) * scores[a] + t * scores[b]
    img = (z @ Vk) + mu
    ax[j].imshow(np.clip(img.reshape(N, N), 0, 1), cmap='magma'); ax[j].axis('off')
fig.suptitle('latent interpolation between two images (smooth morph)'); plt.show()

## Part C — the adversarial idea (a 2D toy GAN)

A minimal illustration of the GAN game in 2D: **real** points come from a target
Gaussian; the **generator** is its own (movable) Gaussian; a logistic
**discriminator** tries to separate real from fake. We alternate — train D to
separate, then nudge G to fool D — and watch the fake cloud drift onto the real
one. (It's a cartoon of the real thing, but the dynamic is genuine.)

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

real_mean, real_sd = np.array([2.5, 1.5]), 0.5     # target distribution
g_mean, g_sd = np.array([-2.0, -2.0]), 0.5         # generator (only g_mean is trained)
w, b = rng.normal(0, 0.1, 2), 0.0                  # logistic discriminator
n = 256
snapshots = {}

for it in range(601):
    real = real_mean + real_sd * rng.standard_normal((n, 2))
    eps  = rng.standard_normal((n, 2))
    fake = g_mean + g_sd * eps
    # --- train D: ascend  E[log D(real)] + E[log(1 - D(fake))] ---
    for _ in range(2):
        pr, pf = sigmoid(real @ w + b), sigmoid(fake @ w + b)
        w += 0.05 * (real.T @ (1 - pr) - fake.T @ pf) / n
        b += 0.05 * ((1 - pr).sum() - pf.sum()) / n
    # --- train G: descend -E[log D(fake)] in g_mean (non-saturating) ---
    pf = sigmoid(fake @ w + b)
    g_mean = g_mean - 0.1 * ((pf - 1)[:, None] * w).mean(0)
    if it in (0, 150, 600):
        snapshots[it] = (real.copy(), fake.copy())

fig, ax = plt.subplots(1, 3, figsize=(14, 4.6))
for a_, it in zip(ax, sorted(snapshots)):
    r, f = snapshots[it]
    a_.scatter(r[:, 0], r[:, 1], s=8, alpha=0.4, label='real')
    a_.scatter(f[:, 0], f[:, 1], s=8, alpha=0.4, label='fake (generator)')
    a_.set_title(f'iteration {it}'); a_.legend(); a_.set_aspect('equal')
    a_.set_xlim(-4, 5); a_.set_ylim(-4, 4)
plt.tight_layout(); plt.show()
print('the fake cloud drifts from (-2,-2) toward the real distribution at (2.5,1.5).')

## Exercises

1. **Anomaly detection (VAE idea):** make a "healthy" set of blobs, fit PCA, then
   feed an odd image (two blobs, or a square). Its reconstruction error should
   spike — that's how VAE anomaly detection works.
2. **Latent dimension:** vary `k` in Part B; too small → blurry/limited samples,
   too large → noisier. Find the sweet spot.
3. **Reparameterisation trick:** sample `z = mu + sigma*eps`, `eps~N(0,1)`, and
   confirm the samples match `N(mu, sigma^2)` — the trick that makes VAEs trainable.
4. **GAN mode collapse:** in Part C, shrink the generator `g_sd` toward 0 and
   watch it collapse to a point — a cartoon of real mode collapse.
5. **Two real modes:** make the real data a mixture of two Gaussians and see the
   single-Gaussian generator struggle to cover both.

## Where this goes (real generative models)

Swap PCA for a deep VAE, the logistic D for a CNN discriminator, and add the
reparameterisation trick + ELBO, and you have the real thing. The MONAI
Generative Models tutorials train VAEs, VQ-GANs, and latent diffusion on real 2D/3D
scans in the `mic_soc` environment. **Project 2** (segmentation) is due this week —
see `project_segmentation.md`.